# House Price Prediction - Model Training

This notebook builds a simple machine learning pipeline and evaluates its performance.


In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pickle

# Load dataset
path = Path('src/notebook/data/raw.csv')
df = pd.read_csv(path)

# Quick cleaning
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date']).dt.year

TARGET = 'price'
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_transformer, numeric_features),
    ('categorical', categorical_transformer, categorical_features),
])

model = LinearRegression()

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

pipeline.fit(X_train, y_train)

train_pred = pipeline.predict(X_train)
test_pred = pipeline.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)
train_rmse = mean_squared_error(y_train, train_pred, squared=False)
test_rmse = mean_squared_error(y_test, test_pred, squared=False)

print('Train R2:', round(train_r2, 4))
print('Test R2:', round(test_r2, 4))
print('Train MAE:', round(train_mae, 2))
print('Test MAE:', round(test_mae, 2))
print('Train RMSE:', round(train_rmse, 2))
print('Test RMSE:', round(test_rmse, 2))

artifacts_dir = Path('artifacts')
artifacts_dir.mkdir(exist_ok=True)

with open(artifacts_dir / 'model.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

print('\nModel saved to artifacts/model.pkl')
